# Select AnimalCLEF clustering parameters

Fits per-model, per-dataset parameters with Optuna using 0.6 × ARI + 0.4 × V-measure. This is the paper's dataset-adapted procedure. Private jaguar is gallery-free and uses fixed defaults.

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("APEXID_REPO_ROOT", Path.cwd())).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "apexid").is_dir():
    raise FileNotFoundError("Start the notebook in the repository or set APEXID_REPO_ROOT.")
sys.path.insert(0, str(REPO_ROOT))

GLOBAL_DESCRIPTOR = os.environ["APEXID_GLOBAL_DESCRIPTOR"]
stage03 = REPO_ROOT / "outputs" / "stage03"
STAGE03A_MODEL_DIR = Path(os.environ.get("APEXID_STAGE03A_OUTPUT", stage03 / "03a")) / GLOBAL_DESCRIPTOR
STAGE03B_MODEL_DIR = Path(os.environ.get("APEXID_STAGE03B_OUTPUT", stage03 / "03b")) / GLOBAL_DESCRIPTOR
STAGE03B_MODEL_DIR.mkdir(parents=True, exist_ok=True)
MATRIX_GLOB = f"sim_matrix_{GLOBAL_DESCRIPTOR}_*_open.npy"

In [ ]:
from collections import defaultdict
import hashlib
import json
import os
import random
from pathlib import Path

import networkx as nx
import numpy as np
import optuna
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.ERROR)
import pandas as pd
from sklearn.metrics import adjusted_rand_score, v_measure_score

# ============================================================
# Optuna knobs (paths: RUN TARGET cell above)
# ============================================================
_REQUIRED = ("STAGE03A_MODEL_DIR", "STAGE03B_MODEL_DIR", "REPO_ROOT", "GLOBAL_DESCRIPTOR", "MATRIX_GLOB")
_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the RUN TARGET cell first; missing: {_missing}")

# Full paper-style run: 2000. Smoke test: set env APEXID_OPTUNA_N_TRIALS=100 (or edit here).
N_TRIALS = int(os.environ.get("APEXID_OPTUNA_N_TRIALS", "2000"))
OPTUNA_SEED = 42
SPLIT_SEED = 42
KNOWN_IDENTITY_FRAC = 0.8
DB_FRAC_WITHIN_KNOWN = 0.8
MIN_IMAGES_PER_KNOWN_IDENTITY = 3
USE_RERANK = True

STAGE03_SPLITS_ROOT_CANDIDATES = (
    Path(os.environ.get("APEXID_STAGE03_SPLITS", stage03 / "03_splits")),
)
QUERY_ONLY_DEFAULTS = {
    "cw_threshold": 0.30,
    "cw_top_k": 20,
    "cw_iterations": 20,
    "cw_k_core_degree": 1,
    "use_rerank": True,
    "rr_k1": 20,
    "rr_k2": 6,
    "rr_lambda": 0.30,
}

ALLOWED_DATASETS = ("bristol_gorillas", "elephants", "gcn_id", "jaguar_antonio")


def parse_open_matrix_filename(path: Path) -> tuple[str, str]:
    name = path.name
    prefix, suffix = "sim_matrix_", "_open.npy"
    if not name.startswith(prefix) or not name.endswith(suffix):
        raise ValueError(f"Not an open-set matrix filename: {name}")
    middle = name[len(prefix) : -len(suffix)]
    for dataset in sorted(ALLOWED_DATASETS, key=len, reverse=True):
        token = f"_{dataset}"
        if middle.endswith(token):
            return middle[: -len(token)], dataset
    raise ValueError(f"Could not parse dataset from {name}; expected one of {ALLOWED_DATASETS}")


def discover_run_jobs(model_dir: Path, output_dir: Path, matrix_glob: str) -> list[dict]:
    jobs = []
    for matrix_path in sorted(model_dir.rglob(matrix_glob)):
        descriptor_from_name, dataset_name = parse_open_matrix_filename(matrix_path)
        jobs.append(
            {
                "matrix_path": matrix_path.resolve(),
                "dataset_name": dataset_name,
                "descriptor_from_filename": descriptor_from_name,
                "output_root": output_dir.resolve(),
            }
        )
    if not jobs:
        raise FileNotFoundError(f"No sim_matrix_*_open.npy in {model_dir}")
    return jobs


RUN_JOBS = discover_run_jobs(STAGE03A_MODEL_DIR, STAGE03B_MODEL_DIR, MATRIX_GLOB)
print("RUN_JOBS:")
for job in RUN_JOBS:
    print(
        f"  - {job['dataset_name']}: {job['matrix_path'].name} -> {job['output_root']}"
    )
print(
    "STAGE03_SPLITS_ROOT:",
    next((p for p in STAGE03_SPLITS_ROOT_CANDIDATES if str(p) and p.is_dir()), "."),
)

In [ ]:
def load_03a_open_bundle(matrix_path: Path, descriptor_from_filename: str, dataset_name: str):
    obj = np.load(matrix_path, allow_pickle=True).item()
    if dataset_name not in obj:
        raise KeyError(f"{matrix_path} has keys {list(obj)}; expected {dataset_name}")
    bundle = obj[dataset_name]
    global_descriptor = str(bundle.get("global_descriptor") or descriptor_from_filename)
    if bundle.get("dataset") != dataset_name:
        raise ValueError("03a dataset does not match filename")
    protocol = str(bundle.get("protocol") or "")
    if protocol not in {"gallery_attach_optuna", "query_only_defaults"}:
        raise ValueError(f"Unknown 03a protocol {protocol!r} in {matrix_path}")

    sim_qq = np.asarray(bundle["sim_query_query"], dtype=np.float32)
    sim_qg = np.asarray(
        bundle.get("sim_query_db", np.zeros((len(sim_qq), 0), dtype=np.float32)),
        dtype=np.float32,
    )
    query_image_ids = np.asarray(bundle["query_image_ids"]).astype(str)
    db_image_ids = np.asarray(bundle["db_image_ids"]).astype(str)
    y_query = np.asarray(bundle["query_identities"]).astype(str)
    y_db = np.asarray(bundle["db_identities"]).astype(str)
    query_open_role = np.asarray(bundle["query_open_role"]).astype(str)

    if sim_qq.shape != (len(query_image_ids), len(query_image_ids)):
        raise ValueError("Query-query matrix does not match query_image_ids")
    if sim_qg.shape != (len(query_image_ids), len(db_image_ids)):
        raise ValueError("Query-gallery matrix does not match query/gallery ids")
    if not np.isfinite(sim_qq).all() or sim_qq.min() < 0 or sim_qq.max() > 1:
        raise ValueError("Query-query similarities must be finite and clipped to [0,1]")
    if sim_qg.size and (not np.isfinite(sim_qg).all() or sim_qg.min() < 0 or sim_qg.max() > 1):
        raise ValueError("Query-gallery similarities must be finite and clipped to [0,1]")
    if len(set(query_image_ids)) != len(query_image_ids) or len(set(db_image_ids)) != len(db_image_ids):
        raise ValueError("03a image_ids are not unique")
    if set(query_image_ids) & set(db_image_ids):
        raise RuntimeError("Query/gallery image leakage in the 03a artifact")

    split = None
    if protocol == "gallery_attach_optuna":
        known_mask = query_open_role == "query_known"
        novel_mask = query_open_role == "query_novel"
        if not known_mask.any() or not novel_mask.any():
            raise RuntimeError("Open split must contain known and novel queries")
        if not len(db_image_ids):
            raise RuntimeError("Standard protocol requires a gallery")
        if set(y_query[novel_mask]) & set(y_db):
            raise RuntimeError("Unknown-query identities leaked into the gallery")
        for identity in np.unique(y_query[known_mask]):
            if identity not in set(y_db):
                raise RuntimeError(f"Known identity {identity!r} lacks a gallery sample")
        split = {
            "sim_query_query": sim_qq,
            "sim_query_db": sim_qg,
            "y_true_query": y_query,
            "y_true_db": y_db,
            "query_image_ids": query_image_ids,
            "db_image_ids": db_image_ids,
            "known_mask_query": known_mask,
            "unknown_mask_query": novel_mask,
        }
    else:
        if sim_qg.shape[1] != 0:
            raise RuntimeError("Gallery-free query-gallery matrix must be empty")

    return {
        "protocol": protocol,
        "global_descriptor": global_descriptor,
        "dataset_name": dataset_name,
        "matrix_path": matrix_path,
        "sim_qq": sim_qq,
        "sim_qg": sim_qg,
        "y_query": y_query,
        "y_db": y_db,
        "split": split,
    }


In [ ]:
from apexid.evaluation.animalclef_open_set import (
    chinese_whispers,
    compute_identity_scores,
    re_ranking,
    resolve_cluster_labels,
    summarize_best_identities,
)

In [ ]:
class _DatasetOptunaProgress:
    """One tqdm bar per dataset (no per-trial Optuna INFO lines)."""

    def __init__(self, total: int, dataset_name: str, initial: int = 0) -> None:
        self._pbar = tqdm(
            total=total,
            initial=initial,
            desc=dataset_name,
            unit="trial",
            dynamic_ncols=True,
            leave=True,
        )

    def __call__(self, study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
        self._pbar.update(1)
        if study.best_trial is not None and study.best_value is not None:
            self._pbar.set_postfix(
                best=f"{study.best_value:.4f}",
                trial=study.best_trial.number,
                refresh=False,
            )

    def close(self) -> None:
        self._pbar.close()


def run_optuna_gallery_attach(split, output_root, global_descriptor, dataset_name):
    sim_qq = split["sim_query_query"]
    sim_qg = split["sim_query_db"]
    y_query = split["y_true_query"]
    y_db = split["y_true_db"]

    def objective(trial):
        top_r = trial.suggest_int("top_r", 1, 5)
        tau_attach = trial.suggest_float("tau_attach", 0.10, 0.90, step=0.01)
        tau_margin = trial.suggest_float("tau_margin", 0.10, 0.90, step=0.01)
        cw_threshold = trial.suggest_float("cw_threshold", 0.12, 0.68, step=0.01)
        cw_top_k = trial.suggest_int("cw_top_k", 7, 60)
        cw_iterations = trial.suggest_int("cw_iterations", 5, 30)
        cw_k_core_degree = trial.suggest_int("cw_k_core_degree", 1, 5)
        rr_k1 = trial.suggest_int("rr_k1", 5, 50)
        rr_k2 = trial.suggest_int("rr_k2", 1, 15)
        rr_lambda = trial.suggest_float("rr_lambda", 0.10, 0.90)

        identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, top_r)
        best_names, best_scores, _, margins = summarize_best_identities(identity_names, identity_scores)
        sim_for_cw = re_ranking(sim_qq, rr_k1, rr_k2, rr_lambda)
        clusters = chinese_whispers(
            sim_for_cw,
            threshold=cw_threshold,
            iterations=cw_iterations,
            top_k=cw_top_k,
            k_core_degree=cw_k_core_degree,
            seed=42,
            skip_mutual_knn=True,
        )
        predicted = resolve_cluster_labels(clusters, best_names, best_scores, margins, tau_attach, tau_margin)
        ari = adjusted_rand_score(y_query, predicted)
        v_measure = v_measure_score(y_query, predicted)
        trial.set_user_attr("ari", float(ari))
        trial.set_user_attr("v_measure", float(v_measure))
        return 0.6 * ari + 0.4 * v_measure

    storage_path = output_root / f"optuna_{global_descriptor}_{dataset_name}.sqlite3"
    study = optuna.create_study(
        study_name=f"{global_descriptor}__{dataset_name}",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED, multivariate=True),
        storage=f"sqlite:///{storage_path.as_posix()}",
        load_if_exists=True,
    )
    completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
    remaining = max(0, N_TRIALS - completed_trials)
    if remaining:
        progress = _DatasetOptunaProgress(N_TRIALS, dataset_name, initial=completed_trials)
        try:
            study.optimize(
                objective,
                n_trials=remaining,
                gc_after_trial=True,
                show_progress_bar=False,
                callbacks=[progress],
            )
        finally:
            progress.close()
    else:
        tqdm.write(f"{dataset_name}: already complete ({completed_trials}/{N_TRIALS} trials)")
    print(
        f"  {dataset_name} done — best={study.best_value:.6f} "
        f"(trial {study.best_trial.number}, {completed_trials + remaining}/{N_TRIALS})"
    )
    return study


def freeze_config_for_job(loaded, study=None):
    protocol = loaded["protocol"]
    global_descriptor = loaded["global_descriptor"]
    dataset_name = loaded["dataset_name"]
    matrix_path = loaded["matrix_path"]
    output_root = loaded["output_root"]
    sim_qq = loaded["sim_qq"]
    sim_qg = loaded["sim_qg"]
    y_query = loaded["y_query"]
    y_db = loaded["y_db"]
    split = loaded["split"]

    split_sha256 = None
    winning_trial_recomputed = False

    if protocol == "gallery_attach_optuna":
        split_path = output_root / f"optuna_split_{global_descriptor}_{dataset_name}.npz"
        np.savez_compressed(split_path, **{k: v for k, v in split.items() if isinstance(v, np.ndarray)})
        split_sha256 = hashlib.sha256(split_path.read_bytes()).hexdigest()

        best_params = dict(study.best_params)
        best_params["use_rerank"] = USE_RERANK
        identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, int(best_params["top_r"]))
        best_names, best_scores, _, margins = summarize_best_identities(identity_names, identity_scores)
        best_sim_for_cw = re_ranking(
            sim_qq,
            int(best_params["rr_k1"]),
            int(best_params["rr_k2"]),
            float(best_params["rr_lambda"]),
        )
        best_clusters = chinese_whispers(
            best_sim_for_cw,
            threshold=float(best_params["cw_threshold"]),
            iterations=int(best_params["cw_iterations"]),
            top_k=int(best_params["cw_top_k"]),
            k_core_degree=int(best_params["cw_k_core_degree"]),
            seed=42,
            skip_mutual_knn=True,
        )
        best_predicted = resolve_cluster_labels(
            best_clusters,
            best_names,
            best_scores,
            margins,
            float(best_params["tau_attach"]),
            float(best_params["tau_margin"]),
        )
        recomputed_ari = float(adjusted_rand_score(y_query, best_predicted))
        recomputed_v_measure = float(v_measure_score(y_query, best_predicted))
        recomputed_objective = 0.6 * recomputed_ari + 0.4 * recomputed_v_measure
        if not np.isclose(recomputed_objective, float(study.best_value), rtol=0.0, atol=1e-12):
            raise RuntimeError(
                f"Winning trial not reproducible for {dataset_name}: "
                f"stored={study.best_value}, recomputed={recomputed_objective}"
            )
        completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
        best_config = {
            **best_params,
            "protocol": protocol,
            "config_source": "optuna_stage03_open_split",
            "objective": "0.6*ARI+0.4*V-measure",
            "objective_value": recomputed_objective,
            "validation_ari": recomputed_ari,
            "validation_v_measure": recomputed_v_measure,
            "n_trials_completed": int(completed_trials),
            "split_seed": SPLIT_SEED,
            "known_identity_frac": KNOWN_IDENTITY_FRAC,
            "db_frac_within_known": DB_FRAC_WITHIN_KNOWN,
            "min_images_per_known_identity": MIN_IMAGES_PER_KNOWN_IDENTITY,
            "dataset": dataset_name,
            "global_descriptor": global_descriptor,
        }
        study.trials_dataframe().to_csv(
            output_root / f"optuna_trials_{global_descriptor}_{dataset_name}.csv", index=False
        )
        winning_trial_recomputed = True
    else:
        best_config = {
            **QUERY_ONLY_DEFAULTS,
            "protocol": protocol,
            "config_source": "frozen_label_free_defaults",
            "tuning_performed": False,
            "n_trials_completed": 0,
            "database_required": False,
            "known_attachment_enabled": False,
            "cw_seed": 42,
            "dataset": dataset_name,
            "global_descriptor": global_descriptor,
        }
        print("  Optuna skipped (query-only defaults):", QUERY_ONLY_DEFAULTS)

    config_path = output_root / f"best_configs_{global_descriptor}_{dataset_name}.json"
    config_path.write_text(json.dumps({dataset_name: best_config}, indent=2, sort_keys=True), encoding="utf-8")
    manifest = {
        "stage": "03b_frozen_clustering_config",
        "status": "passed",
        "protocol": protocol,
        "dataset": dataset_name,
        "global_descriptor": global_descriptor,
        "source_matrix": str(matrix_path),
        "source_matrix_sha256": hashlib.sha256(matrix_path.read_bytes()).hexdigest(),
        "optuna_split_sha256": split_sha256,
        "stage00_test_split_used_for_tuning": False,
        "stage03_open_split_used_for_tuning": protocol == "gallery_attach_optuna",
        "winning_trial_recomputed": winning_trial_recomputed,
        "rerank_mutual_knn_policy": "skip_mutual_knn=True after reranking",
        "config": best_config,
    }
    manifest_path = output_root / f"stage03b_manifest_{global_descriptor}_{dataset_name}.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    print("  Wrote:", config_path.name)
    return config_path


BATCH_SUMMARY = []
for job in RUN_JOBS:
    print("\n" + "=" * 72)
    print("JOB:", job["dataset_name"], "|", job["matrix_path"].name)
    loaded = load_03a_open_bundle(job["matrix_path"], job["descriptor_from_filename"], job["dataset_name"])
    loaded["output_root"] = job["output_root"]
    print("  protocol:", loaded["protocol"], "| descriptor:", loaded["global_descriptor"])
    if loaded["protocol"] == "gallery_attach_optuna":
        print(
            "  query-query:",
            loaded["sim_qq"].shape,
            "query-gallery:",
            loaded["sim_qg"].shape,
            "| gallery:",
            loaded["sim_qg"].shape[1],
        )
        study = run_optuna_gallery_attach(
            loaded["split"],
            job["output_root"],
            loaded["global_descriptor"],
            loaded["dataset_name"],
        )
    else:
        study = None
    config_path = freeze_config_for_job(loaded, study=study)
    BATCH_SUMMARY.append(
        {
            "dataset": loaded["dataset_name"],
            "global_descriptor": loaded["global_descriptor"],
            "protocol": loaded["protocol"],
            "config_path": str(config_path),
        }
    )

print("\n" + "=" * 72)
print(f"Done — {len(BATCH_SUMMARY)} dataset(s) in {STAGE03B_MODEL_DIR}")
for row in BATCH_SUMMARY:
    print(f"  {row['dataset']}: {row['protocol']} -> {row['config_path']}")
print("Next: score with 03c_open_set_metrics.ipynb.")